# Conservation layer for the membrane control paper

Per-residue conservation for the 66 non-transmembrane control proteins: homologue
retrieval, MAFFT alignment, then rate4site under empirical Bayesian JTT.

Results are written one file per protein to `consurf/` in the project folder.


## 1. Mount Drive and set the project folder

Copy this repository to Drive and set `PROJECT` to it. Results are written as they are
computed, so a re-run skips what is already done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Set PROJECT to the folder holding this repository on your Drive.
PROJECT = '/content/drive/MyDrive/Membrane control paper'
OUT     = os.path.join(PROJECT, 'consurf')
PERPROT = os.path.join(OUT, 'per_protein')
WORK    = '/content/work'                                   # scratch, not on Drive

if not os.path.isdir(PROJECT):
    raise SystemExit('Not found: %s' % PROJECT)
DATA = os.path.join(PROJECT, 'data', 'nontm_substitutions.csv.gz')
if not os.path.exists(DATA):
    print('missing: %s  (needed by section 8)' % DATA)
for d in (OUT, PERPROT, WORK): os.makedirs(d, exist_ok=True)
print('project :', PROJECT)
print('outputs :', OUT)
print('existing per-protein results:', len([f for f in os.listdir(PERPROT) if f.endswith('.tsv')]))


## 2. Install the toolchain

`rate4site` and MAFFT, both from the Ubuntu archive.


In [ ]:
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y mafft rate4site > /dev/null 2>&1
!mafft --version 2>&1 | head -1
!which rate4site && rate4site 2>&1 | sed -n '3,5p'


## 3. Parameters


In [ ]:
MAX_HOMOLOGS  = 150      # maximum unique sequences per alignment
TRIM_TO_QUERY_COLUMNS = True   # drop alignment columns where the query is gapped, see section 4
N_WORKERS     = 2        # proteins in parallel; set to the vCPU count
IDENT_LOW     = 0.35     # drop homologues below this identity to the query
IDENT_HIGH    = 0.95     # and above this, as near-duplicates
DEPTH_FLOOR   = 50       # ConSurf's usual reliability floor; below this a protein is flagged
LINSI_MAX     = 150      # use L-INS-i up to this many sequences, --auto above it
R4S_MODEL     = '-Mj'    # JTT, as ConSurf uses
R4S_INFERENCE = '-ib'    # empirical Bayesian (eb-Exp)
OPTIMIZE_BRANCH_LENGTHS = False  # adds rate4site's -bn flag; see section 9 to test it
N_THREADS     = 2        # threads for MAFFT; keep N_WORKERS * N_THREADS near the vCPU count


## 4. Pipeline functions

Homologue retrieval from the UniRef50 cluster, falling back to a Pfam family search across
UniProtKB when the cluster is below the floor. `homolog_source` records the route taken for
each protein, with a `:sp` or `:all` suffix for whether curated entries sufficed.


In [ ]:
import subprocess, time, os, random, json, csv, re, threading
import numpy as np

class FetchError(RuntimeError): pass

# one fetch at a time; UniProt throttles concurrent requests to its stream endpoint
FETCH_LOCK = threading.Semaphore(1)

def curl(url, timeout=300, tries=4):
    """Fetch a URL, or raise. Never returns an empty body silently.

    This matters more than it looks. The first version returned r.stdout unchecked, so a
    throttled or timed-out UniProt request became an empty string, which the caller then
    read as 'this protein has no homologues'. Two of three proteins in a parallel smoke
    test failed that way, and both in fact have over a thousand homologues.
    """
    last = ''
    for k in range(tries):
        with FETCH_LOCK:
            r = subprocess.run(['curl','-sS','--max-time',str(timeout),'-L',
                                '-w','\\n%{http_code}',url], capture_output=True, text=True)
        body, _, code = r.stdout.rpartition('\n')
        if r.returncode == 0 and code.strip() == '200' and body.strip():
            return body
        last = 'rc=%s http=%s %s' % (r.returncode, code.strip(), (r.stderr or '')[:120])
        time.sleep(2 ** k + random.random())      # 1s, 2s, 4s, 8s
    raise FetchError('%s after %d tries: %s' % (url.split('?')[0], tries, last))

def parse_fasta(txt):
    out, name, buf = [], None, []
    for ln in txt.splitlines():
        if ln.startswith('>'):
            if name: out.append((name, ''.join(buf)))
            name, buf = ln[1:].split()[0], []
        elif ln.strip():
            buf.append(ln.strip())
    if name: out.append((name, ''.join(buf)))
    return out

def write_fasta(path, recs):
    with open(path,'w') as f:
        for n,s in recs: f.write('>%s\n%s\n' % (n,s))

def parse_r4s(path):
    """rate4site .res -> [{position, aa, r4s_score, msa_coverage}]. Lower score = more conserved."""
    out = []
    for ln in open(path):
        if ln.startswith('#') or not ln.strip(): continue
        p = ln.split()
        if len(p) >= 3 and p[0].isdigit():
            out.append({'position': int(p[0]), 'aa': p[1], 'r4s_score': float(p[2]),
                        'msa_coverage': p[-1] if '/' in p[-1] else ''})
    return out

def pct_identity(a, b):
    n = m = 0
    for x,y in zip(a,b):
        if x=='-' and y=='-': continue
        n += 1
        if x==y and x!='-': m += 1
    return m/n if n else 0.0

def uniref50_cluster(acc):
    """The UniRef50 cluster that CONTAINS acc, which is not always named after it.

    Querying uniref_cluster_50:UniRef50_<acc> directly only works when the accession is its
    cluster's representative. For Q5JWF2 it returns HTTP 200 with an empty body, no homologues
    and no error, and the protein then falls through to an enormous Pfam family or fails
    outright. Its real cluster is UniRef50_Q6R0H7, with 310 members. Asking UniRef first costs
    about a second and is identical for a protein that IS the representative.
    """
    try:
        j = json.loads(curl('https://rest.uniprot.org/uniref/search'
                            '?query=uniprot_id:%s+AND+identity:0.5&fields=id&format=json' % acc))
        cid = j['results'][0]['id']
        return cid
    except Exception:
        return 'UniRef50_%s' % acc      # fall back to the old assumption

def usable(recs, qseq):
    """Length-filter and deduplicate on exact sequence. This is the count that matters."""
    seen, keep, L = {qseq}, [], len(qseq)
    for n, s in recs:
        if s in seen: continue
        if not (0.5*L <= len(s) <= 1.8*L): continue
        seen.add(s); keep.append((n.replace('|','_'), s))
    return keep

def get_homologs(acc, qseq, floor=DEPTH_FLOOR):
    """Tier 1 UniRef50 cluster; escalate to a Pfam family search if too shallow.

    Two things this gets right that are easy to get wrong.

    First, the escalation decision is made on the number of USABLE sequences, after
    length-filtering and deduplication, not on the raw fetch count. Those differ wildly for
    conserved proteins: RHOA (P61586) returns 124 cluster members, all inside the length
    window, that collapse to 6 unique sequences, because RhoA is near-identical across
    species. GNG2 (P59768) returns 258 and collapses to 3. Testing the raw count means
    escalation never fires for exactly the proteins that need it most, and the bias runs the
    wrong way for this paper: the more conserved the protein, the fewer unique sequences and
    the noisier its conservation estimate.

    Second, a failed fetch is distinguished from a genuinely empty result. Tier 1 failing is
    survivable if tier 2 works, but both failing raises rather than reporting a protein as
    having no homologues.
    """
    best, src, errs = [], 'uniref50', []
    try:
        best = usable(parse_fasta(curl('https://rest.uniprot.org/uniprotkb/stream'
                      '?query=uniref_cluster_50:%s&format=fasta' % uniref50_cluster(acc))), qseq)
    except FetchError as e:
        errs.append(str(e))
    if len(best) < floor:
        try:
            j = json.loads(curl('https://rest.uniprot.org/uniprotkb/%s.json?fields=xref_pfam' % acc))
            pfam = [x['id'] for x in j.get('uniProtKBCrossReferences',[]) if x['database']=='Pfam']
        except Exception as e:
            pfam = []; errs.append('pfam xref: %s' % e)
        # reviewed (Swiss-Prot) entries first, falling back to the full family
        for suffix, tag in (('%20AND%20reviewed:true', 'sp'), ('', 'all')):
            for pf in pfam:
                try:
                    more = usable(parse_fasta(curl('https://rest.uniprot.org/uniprotkb/stream'
                                  '?query=xref:pfam-%s%s&format=fasta' % (pf, suffix))), qseq)
                except FetchError as e:
                    errs.append(str(e)); continue
                if len(more) > len(best):
                    best, src = more, 'pfam:%s:%s' % (pf, tag)
                if len(best) >= floor: break
            if len(best) >= floor: break
    if not best and errs:
        raise FetchError('no homologues retrieved and every request failed: ' + ' | '.join(errs))
    return best, src


In [ ]:
def conservation_for(acc, workdir=None):
    # one working directory per protein: rate4site writes fixed output names
    workdir = os.path.join(workdir or WORK, acc)
    os.makedirs(workdir, exist_ok=True)
    """Full ConSurf-equivalent pipeline for one accession. Returns (rows, log)."""
    t0 = time.time(); log = {'acc': acc}
    q = parse_fasta(curl('https://rest.uniprot.org/uniprotkb/%s.fasta' % acc))
    if not q: return None, dict(log, error='query sequence not retrieved')
    qname, qseq = 'QUERY', q[0][1]
    log['query_length'] = len(qseq)

    # get_homologs already length-filters and deduplicates, and escalates on that count
    pool, src = get_homologs(acc, qseq)
    log['homolog_source'], log['unique_candidates'] = src, len(pool)
    if len(pool) < 5: return None, dict(log, error='too few unique candidate homologues')
    random.Random(0).shuffle(pool)

    # screen in batches, drawing more only when the identity window leaves too few
    BATCH, aln, scored, sel, rounds = MAX_HOMOLOGS*3, {}, {}, [], 0
    t = time.time()
    while rounds*BATCH < len(pool) and rounds < 3:
        batch = pool[rounds*BATCH:(rounds+1)*BATCH]
        rounds += 1
        f1 = os.path.join(workdir, '%s_all%d.fa' % (acc, rounds))
        a1 = os.path.join(workdir, '%s_a%d.fa' % (acc, rounds))
        write_fasta(f1, [(qname,qseq)]+batch)
        subprocess.run('mafft --auto --quiet --thread %d %s > %s' % (N_THREADS,f1,a1), shell=True)
        b = dict(parse_fasta(open(a1).read()))
        if qname not in b: continue
        qa = b[qname]
        for n in b:
            if n == qname: continue
            i = pct_identity(qa, b[n])
            if IDENT_LOW <= i <= IDENT_HIGH:
                aln[n], scored[n] = b[n].replace('-',''), i
        sel = list(scored)
        if len(sel) >= MAX_HOMOLOGS: break
    log['t_align_screen'] = round(time.time()-t,1)
    log['screening_rounds'] = rounds
    log['candidates_screened'] = min(rounds*BATCH, len(pool))
    if not sel: return None, dict(log, error='no candidate fell inside the identity window')
    log['in_identity_window'] = len(sel)
    if len(sel) > MAX_HOMOLOGS:
        sel = sorted(sel, key=lambda n: scored[n])
        step = len(sel)/MAX_HOMOLOGS
        sel = [sel[int(i*step)] for i in range(MAX_HOMOLOGS)]   # spread across the identity range
    log['homologs_used'] = len(sel)
    log['below_depth_floor'] = len(sel) < DEPTH_FLOOR
    if len(sel) < 5: return None, dict(log, error='too few homologues after filtering')

    f2 = os.path.join(workdir, acc+'_sel.fa')
    write_fasta(f2, [(qname,qseq)] + [(n, aln[n]) for n in sel])   # aln holds ungapped sequence
    a2 = os.path.join(workdir, acc+'_msa.fa')
    mode = '--maxiterate 1000 --localpair' if len(sel) <= LINSI_MAX else '--auto'
    t = time.time()
    subprocess.run('mafft %s --quiet --thread %d %s > %s' % (mode,N_THREADS,f2,a2), shell=True)
    log['t_align_final'], log['mafft_mode'] = round(time.time()-t,1), mode

    log['mean_identity_to_query'] = round(float(np.mean([scored[n] for n in sel])), 4)

    # trim to columns where the query has a residue; only query positions are scored
    fin = dict(parse_fasta(open(a2).read()))
    if qname not in fin: return None, dict(log, error='query lost during final alignment')
    log['columns_before_trim'] = len(fin[qname])
    if TRIM_TO_QUERY_COLUMNS:
        cols = [i for i,ch in enumerate(fin[qname]) if ch != '-']
        a3 = os.path.join(workdir, acc+'_msa_trim.fa')
        write_fasta(a3, [(n, ''.join(s[i] for i in cols)) for n,s in fin.items()])
        a2 = a3
        log['columns_after_trim'] = len(cols)

    res = os.path.join(workdir, acc+'.r4s')
    bn  = '' if OPTIMIZE_BRANCH_LENGTHS else '-bn'
    cmd = ('cd %s && rate4site -s %s -o %s -a %s %s %s %s 2>&1'
           % (workdir,a2,res,qname,R4S_INFERENCE,R4S_MODEL,bn))
    # retry once on an empty output
    t = time.time()
    for attempt in (1, 2):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if os.path.exists(res) and os.path.getsize(res) > 0: break
        log['rate4site_attempts'] = attempt + 1
    log['t_rate4site'], log['rate4site_rc'] = round(time.time()-t,1), r.returncode
    if not os.path.exists(res) or os.path.getsize(res) == 0:
        return None, dict(log, error='rate4site produced no output in 2 attempts',
                          stderr=r.stdout[-300:])

    rows = parse_r4s(res)
    if not rows:
        return None, dict(log, error='rate4site produced no scored positions')

    # keep r4sOrig.res (unnormalised rates) and TheTree.txt before they are overwritten
    orig = os.path.join(workdir,'r4sOrig.res')
    if os.path.exists(orig):
        unn = {d['position']: d['r4s_score'] for d in parse_r4s(orig)}
        for d in rows: d['r4s_rate_unnorm'] = unn.get(d['position'])
        os.replace(orig, os.path.join(workdir, acc+'_orig.r4s'))
    tre = os.path.join(workdir,'TheTree.txt')
    if os.path.exists(tre):
        log['tree_length'] = round(sum(float(x) for x in re.findall(r':([0-9.eE+-]+)', open(tre).read())), 3)
        os.replace(tre, os.path.join(workdir, acc+'_tree.txt'))
    m = re.search(r'alpha parameter ([0-9.]+)', open(res).read())
    if m: log['gamma_alpha'] = float(m.group(1))

    log['residues_scored'] = len(rows)
    log['t_total'] = round(time.time()-t0,1)
    # keep the alignment
    return rows, log

print('pipeline defined')


## 5. The 66 control proteins

Read from `consurf_worklist.csv` in the project folder if present, otherwise from the list
below.


In [ ]:
FREE_PEPTIDE = ['O00230','P01185','P01189','P01215','P01222','P01225','P01275','P01286','P01298',
                'P01350','P05305','P06307','P07492','P09681','P10145','P18509','P22466','P35318',
                'P48061','P55089','P78556','Q96A98','Q9H1J5','Q9UHF0','Q9Y5Q6']
MEMB_COMPLEX = ['O43681','O75348','O75947','O75964','O95721','P04899','P06576','P08754','P09471',
                'P18859','P21281','P21283','P21912','P24539','P25705','P30049','P31040','P36542',
                'P36543','P38405','P38606','P48047','P50148','P56381','P56385','P59768','P61421',
                'P61586','P62072','P62873','P63096','Q13303','Q14344','Q16864','Q53H12','Q5JWF2',
                'Q9UI12','Q9UII2','Q9Y3B6','Q9Y5J7','Q9Y5K8']
STRATUM = {a:'free_peptide' for a in FREE_PEPTIDE}
STRATUM.update({a:'memb_complex' for a in MEMB_COMPLEX})
ACCS = FREE_PEPTIDE + MEMB_COMPLEX
assert len(ACCS) == 66 and len(set(ACCS)) == 66
print('%d proteins: %d membrane-complex subunits, %d secreted peptides'
      % (len(ACCS), len(MEMB_COMPLEX), len(FREE_PEPTIDE)))


## 6. Run

A protein already written to Drive is skipped. Proteins run `N_WORKERS` at a time; rate4site
is single-threaded, so set `N_WORKERS` to the vCPU count.


In [ ]:
import pandas as pd, json, os
# threads, not processes: the heavy work runs in subprocesses that release the GIL
from concurrent.futures import ThreadPoolExecutor, as_completed
LOGFILE = os.path.join(OUT, 'run_log.jsonl')

def write_result(acc, rows, log):
    log['stratum'] = STRATUM[acc]
    with open(LOGFILE,'a') as f: f.write(json.dumps(log)+'\n')
    if rows is None: return False
    df = pd.DataFrame(rows)
    df.insert(0,'uniprot',acc); df.insert(1,'stratum',STRATUM[acc])
    df['homologs_used']     = log['homologs_used']
    df['below_depth_floor'] = log['below_depth_floor']
    df['homolog_source']    = log['homolog_source']
    tmp = os.path.join(PERPROT, acc+'.tsv.part')          # write-then-rename, so an
    df.to_csv(tmp, sep='\t', index=False)                 # interrupted write is never
    os.replace(tmp, os.path.join(PERPROT, acc+'.tsv'))    # mistaken for a finished protein
    return True

todo = [a for a in ACCS if not os.path.exists(os.path.join(PERPROT, a+'.tsv'))]
print('already done: %d, to do: %d, %d at a time\n' % (len(ACCS)-len(todo), len(todo), N_WORKERS))

n = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    fut = {ex.submit(conservation_for, acc): acc for acc in todo}
    for f in as_completed(fut):
        acc = fut[f]; n += 1
        try:
            rows, log = f.result()
        except Exception as e:
            with open(LOGFILE,'a') as fh:
                fh.write(json.dumps({'acc':acc,'stratum':STRATUM[acc],
                                     'error':'%s: %s' % (type(e).__name__, e)})+'\n')
            print('%3d/%d  %-8s CRASHED  %s' % (n, len(todo), acc, e)); continue
        if not write_result(acc, rows, log):
            print('%3d/%d  %-8s FAILED   %s' % (n, len(todo), acc, log.get('error'))); continue
        print('%3d/%d  %-8s %4d residues  %3d homologues (%s)  %4d cols  %6.1fs%s'
              % (n, len(todo), acc, log['residues_scored'], log['homologs_used'],
                 log['homolog_source'], log.get('columns_after_trim', log.get('columns_before_trim', 0)),
                 log['t_total'], '  [BELOW DEPTH FLOOR]' if log['below_depth_floor'] else ''))
print('\ndone')


## 7. Assemble the master table and run QC

`grade_within_protein` is the 1 to 9 bin, 9 most conserved; the raw `r4s_score` runs the
other way. Rates are standardised inside each alignment, so the scores are within-protein.


In [ ]:
import numpy as np, pandas as pd, glob, json, os
LOGFILE = os.path.join(OUT, 'run_log.jsonl')
parts = [pd.read_csv(f, sep='\t') for f in sorted(glob.glob(os.path.join(PERPROT,'*.tsv')))]
master = pd.concat(parts, ignore_index=True)

master['conservation'] = -master.r4s_score      # higher = more conserved, for convenience

# grades are assigned within each protein; 9 = most conserved
def grade(s):
    if s.nunique() < 9: return pd.Series(np.full(len(s), np.nan), index=s.index)
    return 9 - pd.qcut(s, 9, labels=False, duplicates='drop')
master['grade_within_protein'] = (master.groupby('uniprot').r4s_score
                                        .transform(grade).astype('Int64'))
# pooled bin, for plotting only; not a cross-protein conservation scale
master['grade_pooled'] = (9 - pd.qcut(master.r4s_score, 9, labels=False, duplicates='drop')).astype(int)
master.to_csv(os.path.join(OUT,'All_Controls_Conservation_Master.tsv'), sep='\t', index=False)
print('master:', master.shape, '->', os.path.join(OUT,'All_Controls_Conservation_Master.tsv'))

qc = (master.groupby(['uniprot','stratum'])
            .agg(residues=('position','size'),
                 homologs=('homologs_used','first'),
                 below_floor=('below_depth_floor','first'),
                 source=('homolog_source','first')).reset_index())
# attach the per-protein divergence diagnostics from the run log
lg = pd.DataFrame([json.loads(l) for l in open(LOGFILE)]) if os.path.exists(LOGFILE) else pd.DataFrame()
for col in ('mean_identity_to_query','tree_length','gamma_alpha'):
    if col in lg.columns:
        qc[col] = qc.uniprot.map(lg.drop_duplicates('acc', keep='last').set_index('acc')[col])
qc.to_csv(os.path.join(OUT,'conservation_qc.csv'), index=False)

print('\n=== QC: homologue depth by stratum ===')
print(qc.groupby('stratum').homologs.describe()[['count','min','25%','50%','75%','max']].to_string())
for col in ('mean_identity_to_query','tree_length','gamma_alpha'):
    if col in qc.columns and qc[col].notna().any():
        print('\n%s by stratum:' % col)
        print(qc.groupby('stratum')[col].describe()[['count','min','50%','max']].to_string())
print('\nproteins below the depth floor of %d:' % DEPTH_FLOOR)
print(qc.groupby('stratum').below_floor.agg(['sum','size']).to_string())
bad = qc[qc.below_floor]
if len(bad):
    print('\n', bad[['uniprot','stratum','homologs','source']].to_string(index=False))
print('\nif the two strata differ substantially in depth, run section 8 both ways')


## 8. Conservation-controlled effect sizes

Per-protein effect sizes computed within conservation quintiles defined inside each protein,
on all 66 proteins and on the subset above the depth floor. Reads
`data/nontm_substitutions.csv.gz` from the project folder.


In [ ]:
import pandas as pd, numpy as np, os
from scipy.stats import mannwhitneyu, wilcoxon

NONPOLAR, POLAR = set('ACFGILMPVW'), set('DEHKNQRSTY')
src = os.path.join(PROJECT,'data','nontm_substitutions.csv.gz')
assert os.path.exists(src), 'copy the paper folder into Drive first; missing ' + src
d = pd.read_csv(src)

cons = master[['uniprot','position','conservation','grade_within_protein',
               'below_depth_floor','homolog_source']]
m = d.merge(cons, left_on=['acc','position'], right_on=['uniprot','position'], how='inner')
print('substitutions with conservation attached: %d of %d (%.1f%%)'
      % (len(m), len(d), 100*len(m)/len(d)))

m['dir'] = np.where(m.aa1.isin(NONPOLAR) & m.aa2.isin(POLAR), 'np',
            np.where(m.aa1.isin(POLAR) & m.aa2.isin(NONPOLAR), 'pn', 'x'))

def per_protein(df, adjust):
    out = []
    for acc, g in df.groupby('acc'):
        if adjust:                      # compare within conservation deciles, then pool
            g = g.copy()
            g['bin'] = pd.qcut(g.conservation, 5, labels=False, duplicates='drop')
            ds = []
            for _, gg in g.groupby('bin'):
                a, b = gg[gg['dir']=='np'].score, gg[gg['dir']=='pn'].score
                if len(a) >= 10 and len(b) >= 10:
                    u,_ = mannwhitneyu(a,b); ds.append(2*u/(len(a)*len(b))-1)
            if not ds: continue
            delta = float(np.mean(ds))
        else:
            a, b = g[g['dir']=='np'].score, g[g['dir']=='pn'].score
            if len(a) < 20 or len(b) < 20: continue
            u,_ = mannwhitneyu(a,b); delta = 2*u/(len(a)*len(b))-1
        out.append({'acc':acc,'cls':g['class'].iloc[0],'delta':delta,
                    'below_floor':bool(g.below_depth_floor.iloc[0])})
    return pd.DataFrame(out)

CUTS = [('all',        'ALL 66 PROTEINS',                m),
        ('abovefloor', 'ABOVE DEPTH FLOOR ONLY',         m[~m.below_depth_floor]),
        ('uniref50',   'UNIREF50 HOMOLOGUES ONLY (no Pfam escalation)',
                                                          m[m.homolog_source == 'uniref50'])]
for key, label, subset in CUTS:
    print('\n' + '='*68); print(label)
    for adjust in (False, True):
        pp = per_protein(subset, adjust)
        if pp.empty: print('  no proteins met the minimum counts'); continue
        tag = 'conservation-ADJUSTED' if adjust else 'unadjusted'
        print('  %s' % tag)
        for cl in ('memb_complex','free_peptide'):
            s = pp[pp.cls==cl]
            if len(s) < 3: continue
            w,p = wilcoxon(s.delta, np.zeros(len(s)))
            print('    %-13s n=%2d  median %+0.4f  positive in %2d/%2d  Wilcoxon p=%.3g'
                  % (cl, len(s), s.delta.median(), int((s.delta>0).sum()), len(s), p))
        a = pp[pp.cls=='memb_complex'].delta; b = pp[pp.cls=='free_peptide'].delta
        if len(a) >= 3 and len(b) >= 3:
            u,p = mannwhitneyu(a,b,alternative='two-sided')
            print('    between strata: difference in medians %+0.4f,  p = %.3g'
                  % (a.median()-b.median(), p))
        pp.to_csv(os.path.join(OUT,'effects_%s_%s.csv'
                  % (key, 'adj' if adjust else 'raw')), index=False)
print('\nper-protein effect tables written to', OUT)


## 9. Optional follow-ups
**1. Branch-length sensitivity.** Re-runs ten proteins with branch-length optimisation on.

```python
import random, shutil, pandas as pd
sample = random.Random(0).sample(ACCS, 10)
OPTIMIZE_BRANCH_LENGTHS = True
rows = []
for acc in sample:
    fast = pd.read_csv(os.path.join(PERPROT, acc+'.tsv'), sep='\t')
    shutil.rmtree(os.path.join(WORK, acc), ignore_errors=True)
    r, lg = conservation_for(acc)
    if r is None: continue
    slow = pd.DataFrame(r)
    j = fast.merge(slow, on='position', suffixes=('_fast','_full'))
    g = lambda s: (9 - pd.qcut(s, 9, labels=False, duplicates='drop'))
    a, b = g(j.r4s_score_fast), g(j.r4s_score_full)
    rows.append({'acc': acc, 'n': len(j),
                 'rho': j.r4s_score_fast.corr(j.r4s_score_full, method='spearman'),
                 'identical_grade': (a == b).mean(), 'within_one': (abs(a-b) <= 1).mean()})
    print(rows[-1])
pd.DataFrame(rows).to_csv(os.path.join(OUT,'branch_length_sensitivity.csv'), index=False)
```

**2. Keep the alignments.** They are not copied to Drive by default.

```python
import glob, shutil, os
dst = os.path.join(OUT, 'alignments'); os.makedirs(dst, exist_ok=True)
# each protein has its own subdirectory under WORK
src = glob.glob(os.path.join(WORK, '*', '*_msa.fa'))
for f in src: shutil.copy2(f, dst)
print('copied %d alignments to %s' % (len(src), dst))
```
